In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import pandas as pd
from pathlib import Path
%matplotlib inline

# 09 - Intensity Profiling Research

Kutatas: kulonbozo 1D profil-strategiak osszehasonlitasa a kanonikus ROI-n (600x80 px).

## Cel

Megvizsgaljuk, melyik oszloponkenti aggregacio kuloniti el legjobban a bundokat a zajtol,
kulonosen dolt (shear-olt) fretboardokon -- melyik modszer "omlik ossze" legkesobb.

## Vizsgalt strategiak

| # | Modszer | Formula | Elmelet |
|---|---|---|---|
| 1 | **Linearis (Baseline)** | `mean(gray, axis=0)` | Jelenlegi referencia; zaj atlagolodik |
| 2 | **Negyzetes (Power x2)** | `mean(gray^2, axis=0)` | Bunteti a sotet zajt, felerositia csucsokat |
| 3 | **Max-pooling** | `max(gray, axis=0)` | Dolesre-invarians: csak a legfenyesebb pixel |
| 4 | **Sobel-X** (jelenlegi motor) | `sum(|dI/dx|, axis=0)` | Eldetektals: csak vertikalis eleket lat |

**Hipotezis**: a Max-pooling profil leginvarianosabb a shear-re, mert `max()` nem erzekeny
arra, hogy a fenycSik *melyik sorban* bukkan fel.

## Struktura

1. Profilozofuggvenyek definialasa
2. Multi-image diagnosztikai galeria (ROI | Profilok | Kuszob-maszkok)
3. Shear-szimulacio: progressziv doles es SNR-degradacio
4. SNR vs. szog gorbe osszehasonlitasa
5. Kovetkztetesek + kandidalo kod (`src/fretboard.py`-ba emeleshez)

In [ ]:
# ==================================================================
#  GLOBALIS KONFIGURACIOEFIGURÁCIÓ
# ==================================================================

FIG_DPI        = 120
SMOOTH_SIGMA   = 1.5    # Gaussian simitas sigma
POWER_EXP      = 2.0    # kitevo a negyzetes profilhoz
THRESHOLD      = 0.30   # normalizalt kuszob a maszk-vizualizaciohoz

PROFILE_NAMES  = ["Linearis", "Negyzetes (x^2)", "Max-pool", "Sobel-X"]
PROFILE_COLORS = ["#3498db",  "#e74c3c",          "#2ecc71",  "#f39c12"]

OUTPUT_DIR = Path("..") / "output" / "09_intensity_profiling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Strategiak: {PROFILE_NAMES}")

In [ ]:
# ==================================================================
#  ADATBETOLTES
# ==================================================================
from src.dataset import load_manifest
from src.geometry import bgr2rgb, load_image_bgr
from src.fretboard import run_v14_pipeline
from src.preprocess import ImagePreprocessor
from src.config import PREPROCESSING_CONFIG

SAMPLE_PER_CLASS = 2
MAX_IMAGES       = 8
RANDOM_STATE     = 17

manifest = load_manifest()
frames = []
for cls, grp in manifest.groupby("class"):
    n = min(SAMPLE_PER_CLASS, len(grp))
    frames.append(grp.sample(n, random_state=RANDOM_STATE))
test_df = pd.concat(frames).reset_index(drop=True)
if MAX_IMAGES:
    test_df = test_df.head(MAX_IMAGES)

prep = ImagePreprocessor(config=PREPROCESSING_CONFIG)
print(f"Tesztképek: {len(test_df)}  |  osztályok: {sorted(test_df['class'].unique())}")

# Pipeline elorefuttatasa
print("Pipeline futtatasa...")
pipeline_results = []
for _, row in test_df.iterrows():
    img_path = Path(row["path"])
    res = run_v14_pipeline({"path": img_path, "class": row["class"]},
                           preprocessor=prep)
    pipeline_results.append(res)
    status = "OK" if res.get("ok") else f"FAIL ({res.get('invalid_reason', '?')})"
    print(f"  [{row['class']}] {img_path.name}  -> {status}")

valid_results = [(row, res) for (_, row), res in
                 zip(test_df.iterrows(), pipeline_results)
                 if res.get("ok") and res.get("canon") is not None]
print(f"\nErvényes: {len(valid_results)} / {len(test_df)}")

---

## 1. Profilozó Stratégiák

Mindegyik függvény ugyanabból a `canon_bgr` (600x80 px BGR) képből állítja elő
a normalizált [0,1] értékű 1D profilt. Azonos Gaussian simítás (`SMOOTH_SIGMA`) alkalmazódik
az összehasonlíthatóság érdekében.

In [ ]:
# ==================================================================
#  1D PROFIL-STRATEGIAK
# ==================================================================

def _gray(roi_bgr):
    return cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)

def _norm(arr):
    m = float(arr.max())
    return (arr / m).astype(np.float32) if m > 1e-6 else arr.astype(np.float32)

def _smooth(arr, sigma):
    return gaussian_filter1d(arr, sigma) if sigma > 0 else arr


def profile_linear(roi_bgr, sigma=SMOOTH_SIGMA):
    """Baseline: oszloponkenti atlagos szurkeintenzitas."""
    return _norm(_smooth(_gray(roi_bgr).mean(axis=0), sigma))


def profile_power(roi_bgr, power=POWER_EXP, sigma=SMOOTH_SIGMA):
    """Power transform: (gray/255)^power atlag -- felerositia a csucsokat."""
    g = _gray(roi_bgr) / 255.0
    return _norm(_smooth((g ** power).mean(axis=0), sigma))


def profile_max(roi_bgr, sigma=SMOOTH_SIGMA):
    """Max-pooling: oszloponkenti maximum -- dolesre-invarians."""
    return _norm(_smooth(_gray(roi_bgr).max(axis=0), sigma))


def profile_sobel(roi_bgr, sigma=SMOOTH_SIGMA, ksize=3):
    """Jelenlegi motor: |Sobel-X| osszeg oszloponkent -- vertikalis eldetektals."""
    sx = cv2.Sobel(_gray(roi_bgr), cv2.CV_32F, 1, 0, ksize=ksize)
    return _norm(_smooth(np.abs(sx).sum(axis=0), sigma))


PROFILERS = [profile_linear, profile_power, profile_max, profile_sobel]


def compute_all_profiles(roi_bgr):
    return [f(roi_bgr) for f in PROFILERS]


print("Profilozó stratégiák:")
for name, fn in zip(PROFILE_NAMES, PROFILERS):
    print(f"  {name:22s}  ->  {fn.__name__}()")

In [ ]:
# ==================================================================
#  build_diagnostic_panel() -- 3-soros diagnosztikai abra
# ==================================================================

def build_diagnostic_panel(roi_bgr, title="", threshold=THRESHOLD,
                           nut_x=None, fret_xs=None):
    """3-soros panel:
    Row 0 -- ROI kep (Nut + Fret vonalak)
    Row 1 -- 4 profil egymásra vetitve + kuszob-vonal
    Row 2 -- Kuszob-alapu binaris maszkok (1 sav / strategia)
    """
    profiles = compute_all_profiles(roi_bgr)
    W  = roi_bgr.shape[1]
    xs = np.arange(W)

    fig = plt.figure(figsize=(14, 7), dpi=FIG_DPI)
    gs  = gridspec.GridSpec(3, 1, height_ratios=[1.4, 2.6, 1.0],
                            hspace=0.08, figure=fig)
    if title:
        fig.suptitle(title, fontsize=10, fontweight="bold", y=1.01)

    # Row 0: ROI kép
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(bgr2rgb(roi_bgr), aspect="auto")
    ax0.set_xlim(0, W - 1)
    ax0.set_xticks([])
    ax0.set_yticks([])
    ax0.set_ylabel("ROI", fontsize=8, rotation=0, labelpad=28)
    if nut_x is not None:
        ax0.axvline(x=nut_x, color="yellow", lw=2.5, label="Nut")
    if fret_xs:
        for i, fx in enumerate(fret_xs):
            ax0.axvline(x=fx, color="#2ecc71", lw=1.5, alpha=0.85,
                        label="Fret" if i == 0 else "")
    if nut_x is not None or fret_xs:
        ax0.legend(fontsize=7, loc="upper right")

    # Row 1: Profilok
    ax1 = fig.add_subplot(gs[1])
    for prof, name, color in zip(profiles, PROFILE_NAMES, PROFILE_COLORS):
        ax1.plot(xs, prof, color=color, linewidth=1.6, label=name, alpha=0.88)
    ax1.axhline(y=threshold, color="gray", linestyle=":", lw=1.2,
                label=f"kuszob = {threshold}")
    if nut_x is not None:
        ax1.axvline(x=nut_x, color="yellow", lw=2, linestyle="--", alpha=0.8)
    if fret_xs:
        for fx in fret_xs:
            ax1.axvline(x=fx, color="#2ecc71", lw=0.9, alpha=0.5)
    ax1.set_xlim(0, W - 1)
    ax1.set_ylim(0, 1.08)
    ax1.set_xticks([])
    ax1.set_ylabel("Norm. profil", fontsize=8)
    ax1.legend(fontsize=7.5, loc="upper right", ncol=3, framealpha=0.85)
    ax1.grid(alpha=0.22)

    # Row 2: Kuszob-maszkok
    ax2 = fig.add_subplot(gs[2])
    n_prof   = len(profiles)
    mask_img = np.zeros((n_prof, W, 3), dtype=np.uint8)
    for ri, (prof, color) in enumerate(zip(profiles, PROFILE_COLORS)):
        mask = prof >= threshold
        rgb  = tuple(int(c * 255) for c in mcolors.to_rgb(color))
        mask_img[ri, mask]  = rgb
        mask_img[ri, ~mask] = (18, 18, 18)
    ax2.imshow(mask_img, aspect="auto", interpolation="nearest")
    ax2.set_yticks(range(n_prof))
    ax2.set_yticklabels(PROFILE_NAMES, fontsize=7)
    ax2.set_xlabel("Kanonikus x [px]", fontsize=8)
    ax2.set_xlim(0, W - 1)
    if nut_x is not None:
        ax2.axvline(x=nut_x, color="yellow", lw=1.5, linestyle="--")
    if fret_xs:
        for fx in fret_xs:
            ax2.axvline(x=fx, color="white", lw=0.7, alpha=0.4)

    plt.tight_layout()
    return fig


print("build_diagnostic_panel() definialva.")

In [ ]:
# ==================================================================
#  EGYKÉP DEMO
# ==================================================================
if valid_results:
    demo_row, demo_res = valid_results[0]
    demo_canon = demo_res["canon"]
    demo_nut   = demo_res.get("nut")
    demo_frets = demo_res.get("fret_xs_filt", [])
    demo_name  = f"[{demo_row['class']}]  {Path(demo_row['path']).name}"

    fig = build_diagnostic_panel(
        demo_canon,
        title=f"Egykep Demo -- {demo_name}",
        nut_x=int(demo_nut["nut_x"]) if demo_nut else None,
        fret_xs=demo_frets,
    )
    fig.savefig(OUTPUT_DIR / "01_single_image_demo.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"Abra: {OUTPUT_DIR / '01_single_image_demo.png'}")
else:
    print("Nincs ervenyes pipeline eredmeny.")

---

## 2. Multi-image Diagnosztikai Galéria

Minden tesztkep kap egy 3-soros panelt.
A kulonbozo kepek osszehasonlitasa megmutatja, melyik strategia konzisztens.

In [ ]:
# ==================================================================
#  GALERIA LOOP
# ==================================================================
for gi, (row, res) in enumerate(valid_results):
    canon  = res["canon"]
    nut    = res.get("nut")
    frets  = res.get("fret_xs_filt", [])
    title  = f"[{row['class']}]  {Path(row['path']).name}"

    fig = build_diagnostic_panel(
        canon, title=title,
        nut_x=int(nut["nut_x"]) if nut else None,
        fret_xs=frets,
    )
    fp = OUTPUT_DIR / f"gallery_{gi:02d}_{row['class']}_{Path(row['path']).stem}.png"
    fig.savefig(fp, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print(f"[OK] {len(valid_results)} panel megjelenitve.")

---

## 3. Shear-szimuláció: Melyik Profil Robusztusabb?

Egy tokeletes ROI-t szoftveresen megdontunk (`cv2.warpAffine`) 0 deg-tol 8 deg-ig,
es merjuk, hogy az egyes profilok SNR-je hogyan valtozik.

**SNR definicio**: kuszob feletti atlag / kuszob alatti atlag.

**Varható eredmény**: a Max-pooling profil a legrobusztusabb,
mivel `max()` nem erzekeny az el szogdoltesere -- csak azt nezi, hogy
a bund adott x-koordinatan felbukkano pixele elegge fényes-e.

In [ ]:
# ==================================================================
#  SHEAR SZIMULACIO FUGGVENYEK
# ==================================================================

def simulate_shear(canon_bgr, angle_deg):
    """Szoftveresen megdontia a ROI-t -- a bal szel (x=0) rogzitett."""
    h, w = canon_bgr.shape[:2]
    s = np.tan(np.radians(angle_deg))
    M = np.float32([[1., -s, 0.],
                    [0.,  1., 0.]])
    return cv2.warpAffine(canon_bgr, M, (w, h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)


def snr_score(profile, threshold=THRESHOLD):
    """Kuszob feletti atlag / kuszob alatti atlag."""
    above = profile[profile >= threshold]
    below = profile[profile <  threshold]
    if len(above) == 0 or len(below) == 0:
        return 0.0
    return float(above.mean() / (below.mean() + 1e-9))


def peak_count(profile, threshold=THRESHOLD, distance=7):
    """Kuszob feletti csucsok szama."""
    idxs, _ = find_peaks(profile, height=threshold, distance=distance)
    return len(idxs)


def plot_shear_series(canon_bgr, angles, title=""):
    """Shear-szimulalt ROI-k -- 3 sor x N oszlop.
    Row 0: shear-olt ROI kep
    Row 1: 4 profil egymásra vetitve
    Row 2: SNR barchart az adott szoghoz
    Visszater: (Figure, snr_dict)
    """
    n = len(angles)
    fig, axes = plt.subplots(3, n, figsize=(max(3.2 * n, 14), 7), dpi=FIG_DPI,
                             gridspec_kw={"height_ratios": [1, 2, 1.2]})
    if n == 1:
        axes = axes.reshape(3, 1)
    if title:
        fig.suptitle(title, fontsize=10, fontweight="bold")

    snr_dict = {name: [] for name in PROFILE_NAMES}

    for col, angle in enumerate(angles):
        sheared  = simulate_shear(canon_bgr, angle)
        profiles = compute_all_profiles(sheared)

        ax_img = axes[0, col]
        ax_img.imshow(bgr2rgb(sheared), aspect="auto")
        ax_img.set_title(f"a={angle:+.1f} deg", fontsize=9)
        ax_img.axis("off")

        ax_prof = axes[1, col]
        xs = np.arange(sheared.shape[1])
        for prof, name, color in zip(profiles, PROFILE_NAMES, PROFILE_COLORS):
            ax_prof.plot(xs, prof, color=color, lw=1.3, alpha=0.88,
                         label=name if col == 0 else "")
            snr_dict[name].append(snr_score(prof))
        ax_prof.axhline(y=THRESHOLD, color="gray", lw=0.9, linestyle=":")
        ax_prof.set_ylim(0, 1.08)
        ax_prof.set_xlim(0, len(xs) - 1)
        ax_prof.set_xticks([])
        ax_prof.grid(alpha=0.2)
        if col == 0:
            ax_prof.set_ylabel("Norm. profil", fontsize=7)
            ax_prof.legend(fontsize=6.5, loc="upper right")

        ax_snr = axes[2, col]
        snrs = [snr_score(p) for p in profiles]
        ax_snr.bar(range(len(PROFILE_NAMES)), snrs, color=PROFILE_COLORS, alpha=0.85)
        ax_snr.set_xticks(range(len(PROFILE_NAMES)))
        ax_snr.set_xticklabels([n[:4] for n in PROFILE_NAMES], fontsize=6, rotation=30)
        ax_snr.set_ylim(0, max(snrs) * 1.35 + 0.2)
        ax_snr.grid(axis="y", alpha=0.25)
        if col == 0:
            ax_snr.set_ylabel("SNR", fontsize=7)
        for i, snr in enumerate(snrs):
            ax_snr.text(i, snr + 0.02, f"{snr:.2f}", ha="center",
                        va="bottom", fontsize=5.5)

    plt.tight_layout()
    return fig, snr_dict


print("simulate_shear(), snr_score(), peak_count(), plot_shear_series() definialva.")

In [ ]:
# ==================================================================
#  SHEAR SZIMULACIO -- demo kepen
# ==================================================================

SHEAR_ANGLES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0]

if valid_results:
    demo_row, demo_res = valid_results[0]
    demo_canon = demo_res["canon"]
    demo_label = f"[{demo_row['class']}]  {Path(demo_row['path']).name}"

    print(f"Demo: {demo_label}")
    print(f"Szogek: {SHEAR_ANGLES} fok")

    fig_shear, snr_dict = plot_shear_series(
        demo_canon, SHEAR_ANGLES,
        title=f"Shear-szimulacio  --  {demo_label}",
    )
    fig_shear.savefig(OUTPUT_DIR / "02_shear_simulation.png",
                      dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print("\nSNR ertekek szoggenkent:")
    header = "Szog (fok)  " + "".join(f"{n[:8]:>12s}" for n in PROFILE_NAMES)
    print(header)
    print("-" * len(header))
    for ai, angle in enumerate(SHEAR_ANGLES):
        row_str = f"{angle:+6.1f} fok  "
        row_str += "".join(f"{snr_dict[n][ai]:>12.3f}" for n in PROFILE_NAMES)
        print(row_str)
else:
    print("Nincs ervenyes kep.")

In [ ]:
# ==================================================================
#  SNR VS. SZOGFUGGVENY -- vonaldiagram
# ==================================================================

if valid_results and snr_dict:
    fig_snr, axes_snr = plt.subplots(1, 2, figsize=(14, 5), dpi=FIG_DPI)
    fig_snr.suptitle("SNR es Csucsszam degradacioja a shear-szoggel",
                     fontsize=12, fontweight="bold")

    ax_snr = axes_snr[0]
    for name, color in zip(PROFILE_NAMES, PROFILE_COLORS):
        vals = snr_dict[name]
        baseline = vals[0] if vals[0] > 0 else 1.0
        normed   = [v / baseline for v in vals]
        ax_snr.plot(SHEAR_ANGLES, normed, color=color, lw=2.2,
                    marker="o", markersize=5, label=name)
    ax_snr.axhline(y=1.0, color="gray", lw=0.9, linestyle=":")
    ax_snr.axhline(y=0.5, color="gray", lw=0.9, linestyle="--", alpha=0.5,
                   label="50%-os SNR csokkenes")
    ax_snr.set_xlabel("Shear szog [fok]", fontsize=10)
    ax_snr.set_ylabel("Relativ SNR (0 fokhoz kepest)", fontsize=10)
    ax_snr.set_title("SNR degradacio (normalizalt)", fontsize=10)
    ax_snr.legend(fontsize=8)
    ax_snr.grid(alpha=0.3)
    ax_snr.set_xlim(SHEAR_ANGLES[0], SHEAR_ANGLES[-1])

    ax_pk = axes_snr[1]
    for name, color in zip(PROFILE_NAMES, PROFILE_COLORS):
        pk_counts = []
        for angle in SHEAR_ANGLES:
            sh = simulate_shear(demo_canon, angle)
            profs = compute_all_profiles(sh)
            pidx = PROFILE_NAMES.index(name)
            pk_counts.append(peak_count(profs[pidx]))
        ax_pk.plot(SHEAR_ANGLES, pk_counts, color=color, lw=2.2,
                   marker="s", markersize=5, label=name)
    ax_pk.set_xlabel("Shear szog [fok]", fontsize=10)
    ax_pk.set_ylabel("Detektalt csucsok szama", fontsize=10)
    ax_pk.set_title("Csucsszam degradacio", fontsize=10)
    ax_pk.legend(fontsize=8)
    ax_pk.grid(alpha=0.3)
    ax_pk.yaxis.get_major_locator().set_params(integer=True)
    ax_pk.set_xlim(SHEAR_ANGLES[0], SHEAR_ANGLES[-1])

    plt.tight_layout()
    fig_snr.savefig(OUTPUT_DIR / "03_snr_vs_shear.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"Abra: {OUTPUT_DIR / '03_snr_vs_shear.png'}")

---

## 4. Batch SNR -- Minden Tesztkep

Atlagos SNR az osszes ervenyes kepre, 0 fokos shear eseten.
Ez megmutatja, melyik strategia a legjobb allapotan is.

In [ ]:
# ==================================================================
#  BATCH SNR -- osszes tesztkep, 0 fok
# ==================================================================

batch_snr   = {name: [] for name in PROFILE_NAMES}
batch_peaks = {name: [] for name in PROFILE_NAMES}

for row, res in valid_results:
    canon = res["canon"]
    profiles = compute_all_profiles(canon)
    for name, prof in zip(PROFILE_NAMES, profiles):
        batch_snr[name].append(snr_score(prof))
        batch_peaks[name].append(peak_count(prof))

fig_batch, axes_b = plt.subplots(1, 2, figsize=(13, 5), dpi=FIG_DPI)
fig_batch.suptitle(f"Batch SNR -- {len(valid_results)} kep, 0 fokos shear",
                   fontsize=12, fontweight="bold")

x_pos = np.arange(len(PROFILE_NAMES))
means_snr = [np.mean(batch_snr[n])   for n in PROFILE_NAMES]
stds_snr  = [np.std(batch_snr[n])    for n in PROFILE_NAMES]
means_pk  = [np.mean(batch_peaks[n]) for n in PROFILE_NAMES]
stds_pk   = [np.std(batch_peaks[n])  for n in PROFILE_NAMES]

axes_b[0].bar(x_pos, means_snr, yerr=stds_snr, color=PROFILE_COLORS,
              alpha=0.85, capsize=5, width=0.6)
axes_b[0].set_xticks(x_pos)
axes_b[0].set_xticklabels(PROFILE_NAMES, fontsize=8, rotation=15)
axes_b[0].set_ylabel("SNR (atlag +/- std)", fontsize=9)
axes_b[0].set_title("Atlagos SNR (magasabb = jobb)", fontsize=9)
axes_b[0].grid(axis="y", alpha=0.3)
for i, (m, s) in enumerate(zip(means_snr, stds_snr)):
    axes_b[0].text(i, m + s + 0.05, f"{m:.2f}", ha="center", va="bottom", fontsize=8)

axes_b[1].bar(x_pos, means_pk, yerr=stds_pk, color=PROFILE_COLORS,
              alpha=0.85, capsize=5, width=0.6)
axes_b[1].set_xticks(x_pos)
axes_b[1].set_xticklabels(PROFILE_NAMES, fontsize=8, rotation=15)
axes_b[1].set_ylabel("Csucsok szama (atlag +/- std)", fontsize=9)
axes_b[1].set_title("Atlagos detektalt csucsszam", fontsize=9)
axes_b[1].grid(axis="y", alpha=0.3)
for i, (m, s) in enumerate(zip(means_pk, stds_pk)):
    axes_b[1].text(i, m + s + 0.05, f"{m:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
fig_batch.savefig(OUTPUT_DIR / "04_batch_snr.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

print("\n-- SNR osszefoglalo -----------------------------------------")
for name in PROFILE_NAMES:
    m  = np.mean(batch_snr[name])
    s  = np.std(batch_snr[name])
    pk = np.mean(batch_peaks[name])
    print(f"  {name:22s}  SNR={m:.2f}+/-{s:.2f}  peaks={pk:.1f}")

---

## 5. Batch Shear-robusztussag

Minden ervenyes kepre lefuttatjuk a shear szimulaciot 0-5 fokig.
Az atlag +/- std sav mutatja, melyik strategia konzisztensen robusztusabb.
Ez a kulcs abra: statisztikailag szignifikans-e a kulonbseg?

In [ ]:
# ==================================================================
#  BATCH SHEAR-ROBUSZTUSÁG -- összes ervenyes kep
# ==================================================================

SHEAR_ANGLES_BATCH = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]

all_snr = {name: np.zeros((len(valid_results), len(SHEAR_ANGLES_BATCH)))
           for name in PROFILE_NAMES}

for img_i, (row, res) in enumerate(valid_results):
    canon_i = res["canon"]
    for ang_j, angle in enumerate(SHEAR_ANGLES_BATCH):
        sh = simulate_shear(canon_i, angle)
        profs = compute_all_profiles(sh)
        for name, prof in zip(PROFILE_NAMES, profs):
            all_snr[name][img_i, ang_j] = snr_score(prof)

# Normalizalas: minden kep 0 fokos SNR-jehez kepest
all_snr_norm = {}
for name in PROFILE_NAMES:
    baseline = all_snr[name][:, 0:1]  # (N, 1)
    all_snr_norm[name] = all_snr[name] / (baseline + 1e-9)

fig_robust, ax_r = plt.subplots(figsize=(11, 6), dpi=FIG_DPI)
fig_robust.suptitle(
    f"Batch Shear-robusztuság  --  {len(valid_results)} kep atlag +/- std",
    fontsize=12, fontweight="bold",
)

for name, color in zip(PROFILE_NAMES, PROFILE_COLORS):
    m = all_snr_norm[name].mean(axis=0)
    s = all_snr_norm[name].std(axis=0)
    ax_r.plot(SHEAR_ANGLES_BATCH, m, color=color, lw=2.2, marker="o",
              markersize=5, label=name)
    ax_r.fill_between(SHEAR_ANGLES_BATCH, m - s, m + s,
                      color=color, alpha=0.14)

ax_r.axhline(y=1.0, color="gray", lw=0.9, linestyle=":")
ax_r.axhline(y=0.5, color="gray", lw=0.9, linestyle="--", alpha=0.55,
             label="50%-os SNR csokkenes")
ax_r.set_xlabel("Shear szog [fok]", fontsize=11)
ax_r.set_ylabel("Relativ SNR (0 fokhoz normalizalva)", fontsize=11)
ax_r.legend(fontsize=9)
ax_r.grid(alpha=0.28)
ax_r.set_xlim(SHEAR_ANGLES_BATCH[0], SHEAR_ANGLES_BATCH[-1])
ax_r.set_ylim(bottom=0)

plt.tight_layout()
fig_robust.savefig(OUTPUT_DIR / "05_batch_shear_robustness.png",
                   dpi=FIG_DPI, bbox_inches="tight")
plt.show()

print("\n-- Relativ SNR 5 fokos shear-nel --------------------------")
for name in PROFILE_NAMES:
    val = all_snr_norm[name][:, -1].mean()
    rank = "robusztus" if val > 0.8 else ("kozepes" if val > 0.6 else "erzekeniy")
    print(f"  {name:22s}  {val:.3f}  ({rank})")

---

## 6. Következtetések & Kandidáló Kód

### Várható eredmények

| Módszer | Erősség | Gyengeség |
|---|---|---|
| **Lineáris** | Egyszerű, gyors | Zaj is átlagolódik be |
| **Négyzetes (Power)** | Kontrasztosabb csúcsok gyenge zajon | Telített bundoknál nem nyújt többet |
| **Max-pooling** | Leginvariánsabb shear-re | Zaj-max-pixel is átjuthat |
| **Sobel-X** | Legprecízebb egyenes bundoknál | Szögelfordulásra érzékeny |

### Javaslat

- Ha `step6d_shear_correction` jól fut: **Sobel-X** az optimális.
- Ha shear-korrekció hiányzik: **Max-pooling** a legrobusztusabb fallback.
- A `MultiProfileFretDetector` alább plug-and-play módon beépíthető az
  `src/fretboard.py`-ba a `FretDetectorInterface` konvenciót betartva.

In [ ]:
# ==================================================================
#  KANDIDALO KOD -- src/fretboard.py-ba emeleshez
#
#  Plug-and-play: FretDetectorInterface-kompatibilis,
#  cserelhetoa IntensityFretDetector helyett.
# ==================================================================

class MultiProfileFretDetector:
    """Kiserleti bunddetektor: konfiguralható profilstrategiaval.

    mode="linear"  -- np.mean(gray, axis=0)
    mode="power"   -- np.mean((gray/255)^power, axis=0)
    mode="max"     -- np.max(gray, axis=0)
    mode="sobel"   -- |Sobel-X|.sum(axis=0)  [jelenlegi motor]
    mode="auto"    -- SNR-alapu dinamikus valasztas linear/sobel kozott
    """

    def __init__(self, mode="sobel", power=2.0, smooth_sigma=1.5,
                 peak_height=0.12, peak_distance=7, peak_prominence=0.06,
                 peak_max_width=14.0, sobel_ksize=3):
        self.mode            = mode
        self.power           = power
        self.smooth_sigma    = smooth_sigma
        self.peak_height     = peak_height
        self.peak_distance   = peak_distance
        self.peak_prominence = peak_prominence
        self.peak_max_width  = peak_max_width
        self.sobel_ksize     = sobel_ksize

    def _compute_profile(self, canon_bgr):
        gray = cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        if self.mode == "linear":
            p = gray.mean(axis=0)
        elif self.mode == "power":
            p = ((gray / 255.0) ** self.power).mean(axis=0)
        elif self.mode == "max":
            p = gray.max(axis=0)
        elif self.mode in ("sobel", "auto"):
            sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=self.sobel_ksize)
            p  = np.abs(sx).sum(axis=0)
            if self.mode == "auto":
                # Fallback: ha a Sobel SNR gyenge, linear profilt hasznalunk
                p_lin = gray.mean(axis=0)
                snr_s = self._snr(p)
                snr_l = self._snr(p_lin)
                if snr_l > snr_s:
                    p = p_lin
        else:
            raise ValueError(f"Ismeretlen mod: {self.mode!r}")
        if self.smooth_sigma > 0:
            p = gaussian_filter1d(p, self.smooth_sigma)
        mx = float(p.max())
        return (p / mx).astype(np.float32) if mx > 1e-6 else p.astype(np.float32)

    def _snr(self, p, thr=0.30):
        thr_abs = thr * (p.max() + 1e-9)
        above = p[p >= thr_abs]
        below = p[p <  thr_abs]
        return float(above.mean() / (below.mean() + 1e-9)) if len(above) and len(below) else 0.0

    def gradient_profile(self, canon_bgr):
        return self._compute_profile(canon_bgr)

    def detect(self, canon_bgr, nut=None):
        """FretDetectorInterface-kompatibilis detect()."""
        profile = self._compute_profile(canon_bgr)
        peak_idxs, _ = find_peaks(
            profile,
            height=self.peak_height,
            distance=self.peak_distance,
            prominence=self.peak_prominence,
            width=(0.0, self.peak_max_width),
        )
        fret_xs = [float(i) for i in peak_idxs]
        return {
            "fit":           {"method": f"multi_profile_{self.mode}",
                              "coverage_ratio": 0.0, "inlier_count": 0,
                              "nut_anchored": False},
            "fret_xs_raw":   fret_xs,
            "fret_xs_filt":  fret_xs,
            "removed_pairs": [],
            "method":        f"multi_profile_{self.mode}",
            "profile":       profile,
        }


# Gyors teszt
if valid_results:
    canon_test = valid_results[0][1]["canon"]
    print("MultiProfileFretDetector gyors teszt:")
    for mode in ["linear", "power", "max", "sobel"]:
        det   = MultiProfileFretDetector(mode=mode)
        res_t = det.detect(canon_test)
        pk_n  = len(res_t["fret_xs_filt"])
        snr_t = snr_score(res_t["profile"])
        print(f"  mode={mode:8s}  frets={pk_n:2d}  SNR={snr_t:.3f}")
print("\n[OK] MultiProfileFretDetector keszen all src/fretboard.py-ba emelesre.")